# Aclgraph 入图编译与运行

## 前置要求

学习本节前，建议掌握以下基础内容：

- PyTorch Eager 模式的执行方式：算子调用按代码顺序即时触发；
- NPU 计算中 Host 侧与 Device 侧的分工：Host 负责组织和下发任务，Device 负责实际计算；
- `torch.compile` 的基本调用形式；
- 静态 shape、动态 shape、推理循环等基础概念。

## 学习目标

完成本节学习后，能够：

- 理解 Aclgraph 减少 Host 侧重复下发开销的基本原因；
- 区分 Aclgraph 底层能力与 NPUGraph 框架封装之间的关系；
- 说明“捕获、实例化、重放”各阶段的作用；
- 使用 NPUGraph API 显式控制捕获与重放，或通过 `torch.compile` 后端从模型入口启用相关能力；
- 说明 Ascend C 自定义算子通过 Pybind11 或 Torch Library 接入后如何被捕获和重放；
- 通过耗时列表和曲线图观察首次准备成本与后续重放耗时。

## 环境依赖

本节代码支持如下产品型号：

- Ascend 950PR/Ascend 950DT
- Atlas A3 训练系列产品/Atlas A3 推理系列产品
- Atlas A2 训练系列产品/Atlas A2 推理系列产品

运行本节代码前，需要完成 CANN 环境配置，并安装配套版本的 PyTorch 和 `torch_npu`，具体步骤参考 [Ascend Extension for PyTorch 安装](https://www.hiascend.com/document/detail/zh/Pytorch/latest/configandinstg/instg/docs/zh/installation_guide/installation_via_binary_package.md)。第 3.3 节编译自定义算子还需要 CMake 3.16 及以上版本、`make` 和 C++ 编译器。


In [ ]:
# 安装绘图和自定义算子编译所需的 Python 依赖。
%pip install matplotlib numpy pybind11


In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

candidates = []
if os.environ.get("ASCEND_SET_ENV"):
    candidates.append(Path(os.environ["ASCEND_SET_ENV"]))
if os.environ.get("ASCEND_TOOLKIT_HOME"):
    candidates.append(Path(os.environ["ASCEND_TOOLKIT_HOME"]) / "set_env.sh")
if os.environ.get("ASCEND_HOME_PATH"):
    candidates.append(Path(os.environ["ASCEND_HOME_PATH"]) / "set_env.sh")
candidates.append(Path("/usr/local/Ascend/cann/set_env.sh"))

CANN_SET_ENV = next((path for path in candidates if path.is_file()), None)
if CANN_SET_ENV is None:
    raise FileNotFoundError(
        "未找到 CANN set_env.sh，请先安装 CANN Toolkit 或设置 ASCEND_SET_ENV。"
    )

# 将 CANN 环境导入当前 Jupyter 进程，后续启动的终端命令会继承这些变量。
result = subprocess.run(
    ["bash", "-lc", f"source {shlex.quote(str(CANN_SET_ENV))} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.splitlines():
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

OUTPUT_DIR = Path("output/06.02_aclgraph_compile_launch").resolve()
(OUTPUT_DIR / "artifacts").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "results").mkdir(parents=True, exist_ok=True)
os.environ["PY_BIN"] = sys.executable
os.environ["ACLGRAPH_OUTPUT_DIR"] = str(OUTPUT_DIR)

print(f"CANN 环境脚本: {CANN_SET_ENV}")
print(f"Python 解释器: {sys.executable}")
print(f"示例代码目录: {OUTPUT_DIR}")
print("环境初始化完成。")


检查独立 Python 进程能否继承上述环境，并正常导入 PyTorch 与 `torch_npu`。后续示例均使用同一解释器启动，不在已经运行的 Jupyter Python 进程中直接执行 NPU 代码。


In [ ]:
!"$PY_BIN" -c "import torch, torch_npu; \
print('torch:', torch.__version__); \
print('torch_npu:', torch_npu.__version__); \
print('NPU 可用数量:', torch.npu.device_count())"


---

## 1. Aclgraph 的执行原理

PyTorch Eager 模式以算子为调度单位。模型每执行一次，Host 都需要按照执行顺序向 NPU 提交相应任务。对于连续执行的短算子，Host 侧的任务生成、提交和调度开销在总时延中的占比会更加明显；同一计算区域反复运行时，这部分开销也会重复发生。

Aclgraph 针对这一重复下发过程进行优化。捕获阶段，Runtime 记录指定 Stream 上提交的 NPU 任务，以及任务顺序、依赖关系、Kernel 参数和关键内存地址等执行信息。捕获结果经过实例化后形成可执行对象，后续运行只需启动该对象，即可重放已经记录的执行流。由此减少的是 Host 侧重复组织和提交任务的开销，算子的计算逻辑和计算量并未改变。

因此，Aclgraph 中的“图”不是神经网络结构图，也不等同于 PyTorch FX Graph，而是 Runtime 对一段 Device 执行流的表示。Aclgraph 执行对象的生命周期包括三个阶段：

1. **捕获**：记录指定 Stream 上提交的 NPU 任务及其执行关系；
2. **实例化**：根据捕获结果生成可供 Runtime 直接启动的执行对象；
3. **重放**：启动实例化后的执行对象，复用捕获时确定的任务序列。

重放阶段直接复用捕获并实例化后的执行对象，其任务序列和内存关系应保持不变。因此，输入 shape、执行路径和关键内存地址需要保持稳定。使用显式 NPUGraph API 时，新的输入数据通常写入捕获时使用的输入张量，而不是替换为另一块内存地址。

从软件层次看，Aclgraph 是 CANN/ACL Runtime 提供的底层能力。PyTorch 侧的 NPUGraph API 对捕获和重放过程进行了封装，`torch.compile` 的 `npugraphs`、`npugraph_ex` 后端则从模型调用入口组织相关处理。入口形式虽然不同，最终都建立在捕获、实例化和重放机制之上。

<div style="text-align: left; margin: 0; clear: both;"><img src="https://raw.gitcode.com/Ascend/pytorch/raw/v2.7.1-26.0.0/docs/zh/figures/npugraph.png" alt="NPUGraph 优势示意" width="720" style="display: block; margin: 0;"></div>

图中以 NPUGraph 为例展示了 Aclgraph 机制带来的任务下发变化：Eager 模式需要 Host 依次下发算子，捕获重放方式通过一次启动复用已有执行流，从而减少逐项下发产生的调度开销。实际收益取决于 Host 侧开销在总时延中的占比和捕获结果的重放次数；重放次数越多，首次捕获和实例化成本对平均时延的影响越小。


---

## 2. 适用场景

固定 shape、短算子较多且重复执行的低时延计算，更容易通过 Aclgraph 减少 Host 侧重复下发和调度开销。

<table align="left" style="margin-left: 0; margin-right: auto; float: none; text-align: left;">
<thead>
<tr>
<th align="left">场景</th>
<th align="left">是否适合</th>
<th align="left">原因</th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">重复执行的推理循环</td>
<td align="left">适合</td>
<td align="left">同一段计算反复执行，Host 下发开销会累积</td>
</tr>
<tr>
<td align="left">输入形状固定的计算区域</td>
<td align="left">适合</td>
<td align="left">执行形态稳定，捕获结果可以复用</td>
</tr>
<tr>
<td align="left">短算子密集的计算流程</td>
<td align="left">适合</td>
<td align="left">单个算子运行时间较短，Host 下发和调度开销占比更高</td>
</tr>
<tr>
<td align="left">只执行一次的任务</td>
<td align="left">不适合</td>
<td align="left">首次捕获或实例化成本无法被摊薄</td>
</tr>
<tr>
<td align="left">输入形状频繁变化的任务</td>
<td align="left">不适合</td>
<td align="left">捕获结果复用困难，可能频繁重新捕获或重新实例化</td>
</tr>
<tr>
<td align="left">执行路径频繁变化的计算流程</td>
<td align="left">不适合</td>
<td align="left">捕获结果依赖稳定执行路径，频繁变化会降低复用价值</td>
</tr>
</tbody>
</table>
<br clear="all">


---

## 3. 捕获与重放使能方式

面向 PyTorch 模型代码，Aclgraph 相关能力通常通过框架封装入口启用。常用入口可分为两类：一类是 `torch_npu.npu.*` API，用于显式控制捕获和重放；另一类是 `torch.compile` 后端，由框架完成 FX Graph 捕获、后端处理和结果复用。

<table align="left" style="margin-left: 0; margin-right: auto; float: none; text-align: left;">
<thead>
<tr>
<th align="left">路径</th>
<th align="left">常见入口</th>
<th align="left">核心作用</th>
<th align="left">适合场景</th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">API 使能方式</td>
<td align="left"><code>capture_begin()</code> / <code>capture_end()</code>、<code>make_graphed_callables</code>、<code>torch_npu.npu.graph</code></td>
<td align="left">显式控制捕获边界、子模块封装和重放</td>
<td align="left">理解捕获生命周期，或在固定计算区域中显式控制捕获与重放</td>
</tr>
<tr>
<td align="left"><code>torch.compile</code> 使能方式</td>
<td align="left"><code>backend="npugraphs"</code>、<code>backend="npugraph_ex"</code></td>
<td align="left">从模型入口启用编译后端，由框架处理 FX Graph 捕获、编译和运行</td>
<td align="left">固定 shape 推理、模型级后端编译与运行</td>
</tr>
</tbody>
</table>
<br clear="all">

两类入口均依赖 Aclgraph 底层捕获与重放能力，但抽象层次不同：API 入口直接呈现捕获对象及其生命周期，`torch.compile` 入口由框架组织 FX Graph 捕获、后端编译和运行。底层 C/C++ 接口不作为正文路径展开；涉及 Stream、Event、内存生命周期、跨 Stream 捕获或内存地址约束等问题时，可查阅 [CANN Runtime 文档](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition/910beta1/programug/acldevg/runtime_doc_dev_0045.html)。

后续示例采用相同的模型结构、参数、输入数据和计时方法，仅替换捕获与重放入口。统一基线脚本先保存模型参数、输入数据、Eager 输出和耗时；五种执行路径分别在独立 Python 进程中加载这份基线，保证正确性与性能比较采用同一口径。

所有路径使用相同的 `steps`、`batch_size`、`hidden_size` 和 `repeat`。计时时每次调用前后同步 NPU，只有 `npugraphs` 路径额外标记调用边界，以满足该后端连续运行时的处理要求。


In [ ]:
%%writefile output/06.02_aclgraph_compile_launch/common.py
import json
import time
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch_npu


ROOT = Path(__file__).resolve().parent
ARTIFACT_DIR = ROOT / "artifacts"
RESULT_DIR = ROOT / "results"
BASELINE_PATH = ARTIFACT_DIR / "baseline.pt"

CONFIG = {
    "steps": 16,
    "batch_size": 16,
    "hidden_size": 256,
    "repeat": 20,
}


class BenchmarkModel(nn.Module):
    def __init__(self, hidden_size=256, steps=16):
        super().__init__()
        self.steps = steps
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)

    def forward(self, x):
        for _ in range(self.steps):
            residual = x
            x = self.fc1(x)
            x = torch.relu(x)
            x = self.fc2(x)
            x = torch.relu(x + residual)
        return x


def time_model(fn, x, repeat=20, mark_npugraph_step=False):
    times_ms = []
    last_output = None
    with torch.no_grad():
        for _ in range(repeat):
            if mark_npugraph_step:
                torch.compiler.npugraph_mark_step_begin()
            torch_npu.npu.synchronize()
            start = time.perf_counter()
            last_output = fn(x)
            torch_npu.npu.synchronize()
            times_ms.append((time.perf_counter() - start) * 1000)
    return times_ms, last_output


def load_baseline():
    if not BASELINE_PATH.is_file():
        raise FileNotFoundError("请先运行 prepare_baseline.py 生成统一基线。")

    baseline = torch.load(BASELINE_PATH, map_location="cpu")
    config = baseline["config"]
    model = BenchmarkModel(
        hidden_size=config["hidden_size"],
        steps=config["steps"],
    ).npu().eval()
    model.load_state_dict(baseline["model_state"])
    model_input = baseline["input"].npu()
    return model, model_input, baseline


def save_result(name, graph_times, max_error, start_index=0):
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    result = {
        "name": name,
        "graph_times_ms": graph_times,
        "max_error": max_error,
        "start_index": start_index,
    }
    (RESULT_DIR / f"{name}.json").write_text(
        json.dumps(result, indent=2),
        encoding="utf-8",
    )


def plot_timeline(name, eager_times, graph_times, output_name, start_index=0):
    eager_iterations = list(range(1, len(eager_times) + 1))
    graph_values = graph_times[start_index:]
    if start_index == 0:
        graph_iterations = list(range(len(graph_times)))
    else:
        graph_iterations = list(range(1, len(graph_values) + 1))

    plt.figure(figsize=(8, 4.2))
    plt.plot(eager_iterations, eager_times, marker="o", label="Eager")
    plt.plot(graph_iterations, graph_values, marker="o", label=name)
    if start_index == 0:
        plt.scatter(
            [0],
            [graph_times[0]],
            color="#d97706",
            zorder=3,
            label=f"{name} first point",
        )
        plt.axvline(0, color="#9ca3af", linestyle="--", linewidth=1)
        plt.xlabel("Iteration (0 = capture preparation / first call)")
    else:
        plt.xlabel("Iteration")
    plt.ylabel("Time (ms)")
    plt.grid(True, linestyle="--", alpha=0.35)
    plt.legend()
    plt.tight_layout()
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(RESULT_DIR / output_name, dpi=140, bbox_inches="tight")
    plt.close()


def print_summary(label, eager_times, graph_times, max_error, start_index=0):
    stable_times = graph_times[start_index:]
    print(f"max error: {max_error:.6f}")
    if start_index:
        print(f"{label} preparation: {[round(t, 3) for t in graph_times[:start_index]]} ms")
    else:
        print(f"{label} preparation: {graph_times[0]:.3f} ms")
        stable_times = graph_times[1:]
    print(f"Eager baseline avg: {np.mean(eager_times):.3f} ms")
    print(f"{label} stable avg: {np.mean(stable_times):.3f} ms")
    print(f"{label} timeline latency(ms):", [round(t, 3) for t in graph_times])


In [ ]:
%%writefile output/06.02_aclgraph_compile_launch/prepare_baseline.py
import numpy as np
import torch
import torch_npu

from common import ARTIFACT_DIR, BASELINE_PATH, CONFIG, BenchmarkModel, time_model


torch.manual_seed(0)
model_cpu = BenchmarkModel(
    hidden_size=CONFIG["hidden_size"],
    steps=CONFIG["steps"],
).eval()
input_cpu = torch.randn(CONFIG["batch_size"], CONFIG["hidden_size"])

model = model_cpu.npu().eval()
model_input = input_cpu.npu()

# 预热不计入基线耗时，避免首次初始化影响稳定阶段的比较。
with torch.no_grad():
    model(model_input)
torch_npu.npu.synchronize()

eager_times, eager_output = time_model(
    model,
    model_input,
    repeat=CONFIG["repeat"],
)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "config": CONFIG,
        "model_state": model_cpu.state_dict(),
        "input": input_cpu,
        "eager_output": eager_output.cpu(),
        "eager_times": eager_times,
    },
    BASELINE_PATH,
)

print(f"benchmark steps: {CONFIG['steps']}")
print(f"batch size: {CONFIG['batch_size']}")
print(f"hidden size: {CONFIG['hidden_size']}")
print(f"repeat: {CONFIG['repeat']}")
print(f"Eager baseline avg: {np.mean(eager_times):.3f} ms")
print("Eager baseline latency(ms):", [round(t, 3) for t in eager_times])
print(f"baseline artifact: {BASELINE_PATH}")


In [ ]:
!cd "$ACLGRAPH_OUTPUT_DIR" && "$PY_BIN" prepare_baseline.py


### 3.1 API 使能方式：NPUGraph 捕获与重放

`torch_npu.npu` 提供三种常用的 NPUGraph API 形态：手动管理捕获生命周期、封装稳定子模块，以及使用上下文管理器标记捕获区域。三者的控制粒度不同，均用于复用稳定的 NPU 计算区域。

<table align="left" style="margin-left: 0; margin-right: auto; float: none; text-align: left;">
<thead>
<tr>
<th align="left">方法</th>
<th align="left">典型入口</th>
<th align="left">主要作用</th>
<th align="left">适合场景</th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">手动捕获生命周期</td>
<td align="left"><code>NPUGraph()</code>、<code>capture_begin()</code>、<code>capture_end()</code>、<code>replay()</code></td>
<td align="left">显式标记捕获开始和结束，完整呈现捕获对象生命周期</td>
<td align="left">需要完整观察捕获生命周期，或精细控制 Stream 和同步关系</td>
</tr>
<tr>
<td align="left">封装稳定子模块</td>
<td align="left"><code>make_graphed_callables(module, sample_args)</code></td>
<td align="left">将稳定子模块封装成可复用的可调用对象，外围逻辑继续保持 Eager</td>
<td align="left">模型中只有局部区域满足固定 shape 和稳定执行路径条件</td>
</tr>
<tr>
<td align="left">上下文管理器捕获</td>
<td align="left"><code>NPUGraph()</code>、<code>torch_npu.npu.graph(g)</code>、<code>g.replay()</code></td>
<td align="left">用上下文管理器表达捕获边界，减少手动调用捕获起止接口</td>
<td align="left">固定计算区域已经明确，适合观察上下文管理器形式的捕获与重放</td>
</tr>
</tbody>
</table>
<br clear="all">

下面三个示例沿用前文建立的模型结构、参数、输入形状和 Eager 基线，仅改变捕获入口。由于 `make_graphed_callables` 会改写传入模块的 `forward`，对应示例使用参数相同的模型副本。


#### 3.1.1 手动捕获生命周期

手动方式显式调用 `capture_begin()` 和 `capture_end()`，可以直接观察捕获对象从创建、捕获到重复重放的完整生命周期。示例复用 `benchmark_model`，只展开捕获边界和 Stream 同步过程。


In [ ]:
%%writefile output/06.02_aclgraph_compile_launch/manual_capture.py
import time

import torch
import torch_npu

from common import load_baseline, plot_timeline, print_summary, save_result


model, model_input, baseline = load_baseline()
repeat = baseline["config"]["repeat"]
capture_input = model_input.clone()
graph = torch_npu.npu.NPUGraph()
capture_stream = torch_npu.npu.Stream()

# 在捕获 Stream 上完成预热，避免初始化操作进入捕获区域。
capture_stream.wait_stream(torch_npu.npu.current_stream())
with torch.no_grad(), torch_npu.npu.stream(capture_stream):
    for _ in range(3):
        model(capture_input)
torch_npu.npu.current_stream().wait_stream(capture_stream)
torch_npu.npu.synchronize()

graph_times = []
capture_stream.wait_stream(torch_npu.npu.current_stream())
start = time.perf_counter()
with torch.no_grad(), torch_npu.npu.stream(capture_stream):
    graph.capture_begin()
    graph_output = model(capture_input)
    graph.capture_end()
torch_npu.npu.current_stream().wait_stream(capture_stream)
torch_npu.npu.synchronize()
graph_times.append((time.perf_counter() - start) * 1000)

for _ in range(repeat):
    torch_npu.npu.synchronize()
    start = time.perf_counter()
    graph.replay()
    torch_npu.npu.synchronize()
    graph_times.append((time.perf_counter() - start) * 1000)

max_error = (baseline["eager_output"] - graph_output.cpu()).abs().max().item()
print_summary("manual NPUGraph", baseline["eager_times"], graph_times, max_error)
save_result("manual_capture", graph_times, max_error, start_index=1)
plot_timeline(
    "manual NPUGraph",
    baseline["eager_times"],
    graph_times,
    "manual_capture.png",
    start_index=1,
)


In [ ]:
!cd "$ACLGRAPH_OUTPUT_DIR" && "$PY_BIN" manual_capture.py
display(Image(filename=str(OUTPUT_DIR / "results" / "manual_capture.png")))


`graph_times[0]` 表示捕获与实例化耗时，后续元素表示重放耗时。完整耗时列表保留准备阶段，曲线仅绘制重放阶段。手动方式需要调用方管理捕获 Stream 及其同步关系。


#### 3.1.2 封装稳定子模块

`make_graphed_callables` 适合处理只有部分子模块能够稳定复用的模型。稳定子模块被封装成可调用对象，外围分支、日志和 CPU 侧判断仍可保持 Eager。该 API 会改写传入模块的 `forward`，因此示例使用一份参数相同的模型副本。


In [ ]:
%%writefile output/06.02_aclgraph_compile_launch/graphed_callables.py
import time

import torch
import torch_npu

from common import load_baseline, plot_timeline, print_summary, save_result


model, model_input, baseline = load_baseline()
repeat = baseline["config"]["repeat"]
sample_input = model_input.clone()

torch_npu.npu.synchronize()
start = time.perf_counter()
with torch.no_grad():
    graphed_model = torch_npu.npu.make_graphed_callables(model, (sample_input,))
torch_npu.npu.synchronize()
graph_times = [(time.perf_counter() - start) * 1000]

graph_output = None
for _ in range(repeat):
    torch_npu.npu.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        graph_output = graphed_model(model_input)
    torch_npu.npu.synchronize()
    graph_times.append((time.perf_counter() - start) * 1000)

max_error = (baseline["eager_output"] - graph_output.cpu()).abs().max().item()
print_summary("make_graphed_callables", baseline["eager_times"], graph_times, max_error)
save_result("graphed_callables", graph_times, max_error, start_index=1)
plot_timeline(
    "make_graphed_callables",
    baseline["eager_times"],
    graph_times,
    "graphed_callables.png",
    start_index=1,
)


In [ ]:
!cd "$ACLGRAPH_OUTPUT_DIR" && "$PY_BIN" graphed_callables.py
display(Image(filename=str(OUTPUT_DIR / "results" / "graphed_callables.png")))


`graph_times[0]` 表示可调用对象的封装准备耗时，后续元素表示稳定运行耗时。完整耗时列表保留准备阶段，曲线仅绘制稳定运行阶段。该示例从统一基线重新创建模型，因此不会影响其他执行路径。


#### 3.1.3 上下文管理器捕获

上下文管理器将捕获的开始和结束封装到 `with torch_npu.npu.graph(g)` 中，同时保留捕获对象和 `replay()` 入口。固定计算区域已经明确时，可以使用这种写法简化捕获代码。


In [ ]:
%%writefile output/06.02_aclgraph_compile_launch/context_graph.py
import time

import torch
import torch_npu

from common import load_baseline, plot_timeline, print_summary, save_result


model, model_input, baseline = load_baseline()
repeat = baseline["config"]["repeat"]
capture_input = model_input.clone()
graph = torch_npu.npu.NPUGraph()

with torch.no_grad():
    model(capture_input)
torch_npu.npu.synchronize()

start = time.perf_counter()
with torch.no_grad():
    with torch_npu.npu.graph(graph):
        graph_output = model(capture_input)
torch_npu.npu.synchronize()
graph_times = [(time.perf_counter() - start) * 1000]

for _ in range(repeat):
    torch_npu.npu.synchronize()
    start = time.perf_counter()
    graph.replay()
    torch_npu.npu.synchronize()
    graph_times.append((time.perf_counter() - start) * 1000)

max_error = (baseline["eager_output"] - graph_output.cpu()).abs().max().item()
print_summary("NPUGraph replay", baseline["eager_times"], graph_times, max_error)
save_result("context_graph", graph_times, max_error, start_index=1)
plot_timeline(
    "NPUGraph replay",
    baseline["eager_times"],
    graph_times,
    "context_graph.png",
    start_index=1,
)


In [ ]:
!cd "$ACLGRAPH_OUTPUT_DIR" && "$PY_BIN" context_graph.py
display(Image(filename=str(OUTPUT_DIR / "results" / "context_graph.png")))


`graph_times[0]` 记录捕获与实例化耗时，后续元素记录重放耗时。完整耗时列表保留准备阶段，曲线仅绘制重放阶段。输入张量在捕获前创建，后续重放继续使用该地址；更换输入数据时，应将新数据写入原有张量，而不是替换张量对象。

### 3.2 `torch.compile` 使能方式：选择编译后端

`torch.compile` 将模型包装为新的调用入口，不需要显式管理 `NPUGraph` 对象。第一次真实调用通常先由 TorchDynamo 捕获 FX Graph，再交给选定后端处理；后端根据需要完成 Aclgraph 捕获和实例化。后续输入满足复用条件时，已有结果可用于重放。

从 `torch.compile` 入口看，捕获、实例化与重放链路如下：

```text
PyTorch 函数/模型
  -> torch.compile 包装模型调用入口
  -> TorchDynamo 捕获 FX Graph
  -> npugraphs 或 npugraph_ex 后端处理
  -> Aclgraph 捕获并实例化执行对象
  -> 后续稳定输入复用执行对象并重放
```

`npugraphs` 后端从 `torch.compile` 入口使用 NPUGraph 捕获与重放能力；`npugraph_ex` 是 TorchAir 提供的模型级后端，同时处理 FX Graph 优化、编译缓存和内存复用等过程。下面两个示例均复用前文建立的 Eager 基线。

#### 3.2.1 使用 `backend="npugraphs"`


In [ ]:
%%writefile output/06.02_aclgraph_compile_launch/compile_npugraphs.py
import torch

from common import load_baseline, plot_timeline, print_summary, save_result, time_model


model, model_input, baseline = load_baseline()
repeat = baseline["config"]["repeat"]
prepare_count = 2

compiled_model = torch.compile(
    model,
    backend="npugraphs",
    fullgraph=True,
    dynamic=False,
)
graph_times, graph_output = time_model(
    compiled_model,
    model_input,
    repeat=repeat + prepare_count,
    mark_npugraph_step=True,
)

max_error = (baseline["eager_output"] - graph_output.cpu()).abs().max().item()
print_summary(
    "npugraphs",
    baseline["eager_times"],
    graph_times,
    max_error,
    start_index=prepare_count,
)
save_result("compile_npugraphs", graph_times, max_error, start_index=prepare_count)
plot_timeline(
    "npugraphs",
    baseline["eager_times"],
    graph_times,
    "compile_npugraphs.png",
    start_index=prepare_count,
)


In [ ]:
!cd "$ACLGRAPH_OUTPUT_DIR" && "$PY_BIN" compile_npugraphs.py
display(Image(filename=str(OUTPUT_DIR / "results" / "compile_npugraphs.png")))


`backend="npugraphs"` 将模型调用入口交给 NPUGraph 后端处理。该路径在计时函数中设置 `mark_npugraph_step=True`，每次调用前执行 `torch.compiler.npugraph_mark_step_begin()`，标记连续调用之间的边界。API 路径和 `npugraph_ex` 路径不需要该标记。

`graph_times[:prepare_count]` 记录首次准备段，通常包含编译、捕获或实例化准备；其余元素对应后续稳定运行阶段。代码保留并打印完整耗时列表，但曲线只绘制稳定阶段，避免准备段压缩后续运行耗时的观察尺度。

#### 3.2.2 使用 `backend="npugraph_ex"`

`backend="npugraph_ex"` 是 TorchAir 提供的模型级编译后端。该后端通过 `torch.compile` 接入，统一处理 FX Graph 优化、编译缓存和内存复用，并组织 Aclgraph 捕获与重放。

最小配置包含三个关键参数：

```python
torch.compile(model, backend="npugraph_ex", fullgraph=True, dynamic=False)
```

- `backend="npugraph_ex"`：显式选择 npugraph_ex 后端；
- `fullgraph=True`：要求 TorchDynamo 将调用过程捕获为单个 FX Graph，出现 graph break 时直接报错；
- `dynamic=False`：针对当前输入 shape 生成专用结果，输入 shape 变化时可能触发重新编译。


In [ ]:
%%writefile output/06.02_aclgraph_compile_launch/compile_npugraph_ex.py
import torch

from common import load_baseline, plot_timeline, print_summary, save_result, time_model


model, model_input, baseline = load_baseline()
repeat = baseline["config"]["repeat"]
prepare_count = 1

compiled_model = torch.compile(
    model,
    backend="npugraph_ex",
    fullgraph=True,
    dynamic=False,
)
graph_times, graph_output = time_model(
    compiled_model,
    model_input,
    repeat=repeat + prepare_count,
)

max_error = (baseline["eager_output"] - graph_output.cpu()).abs().max().item()
print_summary(
    "npugraph_ex",
    baseline["eager_times"],
    graph_times,
    max_error,
    start_index=prepare_count,
)
save_result("compile_npugraph_ex", graph_times, max_error, start_index=prepare_count)
plot_timeline(
    "npugraph_ex",
    baseline["eager_times"],
    graph_times,
    "compile_npugraph_ex.png",
    start_index=prepare_count,
)


In [ ]:
!cd "$ACLGRAPH_OUTPUT_DIR" && "$PY_BIN" compile_npugraph_ex.py
display(Image(filename=str(OUTPUT_DIR / "results" / "compile_npugraph_ex.png")))


本示例通过 `fullgraph=True` 检查 FX Graph 是否完整，通过 `dynamic=False` 针对当前输入 shape 生成专用结果，使性能对比保持稳定。

该路径同样只替换执行后端，不替换模型、输入或计时方式。`graph_times[0]` 表示第一次真实调用，后续元素表示稳定运行；图中只绘制稳定阶段，`eager_output` 和 `eager_times` 分别作为正确性和耗时参考。

### 3.3 自定义算子的捕获与重放

前文通过 `torch_npu` 适配的框架内置算子说明了 NPUGraph 的捕获与重放机制。当基于 Ascend C 开发的自定义算子通过当前 NPU Stream 启动核函数时，NPUGraph 同样会记录相应任务，因此可以沿用相同的捕获与重放机制。

与框架内置算子不同，基于 Ascend C 开发的自定义算子还需要自行实现算子调用接口，并注册相应的 Python 调用入口：

| 组成部分 | 职责 | 对捕获过程的影响 |
| --- | --- | --- |
| 算子调用接口 | 接收 PyTorch Tensor、准备输出和启动参数，并将核函数任务提交到当前 NPU Stream | 决定哪些 NPU 任务进入捕获范围 |
| Python 接口注册 | 将算子调用接口注册为 Python 可调用对象 | 决定 Python 侧的调用形式，不改变 NPUGraph 捕获机制 |

本节以 `AddCustom` 算子为例，说明 Pybind11 和 Torch Library 两种注册方式。Pybind11 将 C++ 函数直接绑定为 Python 模块函数；Torch Library 将算子定义和 NPU 实现注册到 PyTorch 调度系统。两种方式共用相同的算子调用接口，完整代码参见 [Pybind11 源码](./src/06.02_aclgraph_compile_launch/pybind/) 和 [Torch Library 源码](./src/06.02_aclgraph_compile_launch/torch_library/)。

#### 3.3.1 算子调用接口

样例通过 C++ 函数 `ascendc_add()` 封装对 Ascend C 核函数 `add_custom` 的调用：

```cpp
namespace ascendc_ops {
at::Tensor ascendc_add(const at::Tensor& x, const at::Tensor& y)
{
    auto aclStream = c10_npu::getCurrentNPUStream().stream(false);
    at::Tensor z = at::empty_like(x);
    constexpr uint32_t numBlocks = 8;
    uint32_t totalLength = static_cast<uint32_t>(x.numel());
    add_custom<<<numBlocks, 0, aclStream>>>(
        (uint8_t*)(x.mutable_data_ptr()),
        (uint8_t*)(y.mutable_data_ptr()),
        (uint8_t*)(z.mutable_data_ptr()),
        totalLength);
    return z;
}
} // namespace ascendc_ops
```

代码中的 `ascendc_add()` 接收 PyTorch Tensor，并通过 `getCurrentNPUStream()` 获取当前 NPU Stream；`add_custom<<<...>>>` 使用该 Stream 启动 Ascend C 核函数。调用位于 NPUGraph 捕获区间内时，这次核函数任务会进入捕获的任务序列。

#### 3.3.2 Pybind11 注册

Pybind11 通过函数绑定建立 Python 调用入口：

```cpp
PYBIND11_MODULE(ascendc_ops, m)
{
    m.def("ascendc_add", &ascendc_ops::ascendc_add);
}
```

`PYBIND11_MODULE` 定义 Python 模块 `ascendc_ops`，`m.def()` 将模块函数 `ascendc_add` 绑定到 C++ 调用接口。编译并导入模块后，通过以下形式调用自定义算子：

```python
import ascendc_ops
z = ascendc_ops.ascendc_add(x, y)
```

执行以下代码完成 Pybind11 样例的编译与运行。运行脚本先调用自定义算子完成预热，再捕获并重放该算子，最后校验重放结果。

样例默认编译为 Ascend 950PR/Ascend 950DT 对应的 `dav-3510`；Atlas A2/A3 环境可在运行前将 `ACLGRAPH_ASC_ARCH` 设置为 `dav-2201`。


In [ ]:
CUSTOM_OP_ROOT = Path("src/06.02_aclgraph_compile_launch").resolve()
CUSTOM_OP_BUILD_ROOT = OUTPUT_DIR / "custom_ops"
CUSTOM_OP_ARCH = os.environ.get("ACLGRAPH_ASC_ARCH", "dav-3510")
PYBIND_BUILD_DIR = CUSTOM_OP_BUILD_ROOT / "pybind"

os.environ["ACLGRAPH_CUSTOM_OP_ROOT"] = str(CUSTOM_OP_ROOT)
os.environ["ACLGRAPH_PYBIND_BUILD_DIR"] = str(PYBIND_BUILD_DIR)
os.environ["ACLGRAPH_ASC_ARCH"] = CUSTOM_OP_ARCH

print(f"目标架构: {CUSTOM_OP_ARCH}")

!mkdir -p "$ACLGRAPH_PYBIND_BUILD_DIR"
!cd "$ACLGRAPH_PYBIND_BUILD_DIR" && \
    cmake -DCMAKE_ASC_ARCHITECTURES="$ACLGRAPH_ASC_ARCH" \
          -DPython3_EXECUTABLE="$PY_BIN" \
          "$ACLGRAPH_CUSTOM_OP_ROOT/pybind" && \
    make -j && \
    "$PY_BIN" "$ACLGRAPH_CUSTOM_OP_ROOT/pybind/add_custom_test.py"


#### 3.3.3 Torch Library 注册

Torch Library 通过 PyTorch 算子注册机制建立调用入口：

```cpp
TORCH_LIBRARY(ascendc_ops, m)
{
    m.def("ascendc_add(Tensor x, Tensor y) -> Tensor");
}

TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m)
{
    m.impl("ascendc_add", TORCH_FN(ascendc_ops::ascendc_add));
}
```

`TORCH_LIBRARY` 声明算子名称和输入输出形式，`TORCH_LIBRARY_IMPL` 使用 `PrivateUse1` 调度键注册 NPU 实现。加载动态库后，通过 `torch.ops` 命名空间调用自定义算子：

```python
torch.ops.load_library("libascendc_ops.so")
z = torch.ops.ascendc_ops.ascendc_add(x, y)
```

执行以下代码完成 Torch Library 样例的编译与运行。该样例沿用与 Pybind11 相同的算子调用接口和 NPUGraph 捕获流程。


In [ ]:
TORCH_LIBRARY_BUILD_DIR = CUSTOM_OP_BUILD_ROOT / "torch_library"
os.environ["ACLGRAPH_TORCH_LIBRARY_BUILD_DIR"] = str(TORCH_LIBRARY_BUILD_DIR)

!mkdir -p "$ACLGRAPH_TORCH_LIBRARY_BUILD_DIR"
!cd "$ACLGRAPH_TORCH_LIBRARY_BUILD_DIR" && \
    cmake -DCMAKE_ASC_ARCHITECTURES="$ACLGRAPH_ASC_ARCH" \
          -DPython3_EXECUTABLE="$PY_BIN" \
          "$ACLGRAPH_CUSTOM_OP_ROOT/torch_library" && \
    make -j && \
    "$PY_BIN" "$ACLGRAPH_CUSTOM_OP_ROOT/torch_library/add_custom_test.py"


---

## 4. 官方资料导航

<table align="left" style="margin-left: 0; margin-right: auto; float: none; text-align: left;">
<thead>
<tr>
<th align="left">资料</th>
<th align="left">查阅目的</th>
<th align="left">链接</th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">CANN Runtime：ACL Graph 简介</td>
<td align="left">从 Runtime 视角理解 Stream 任务捕获、执行对象实例化和启动过程</td>
<td align="left"><a href="https://www.hiascend.com/document/detail/zh/CANNCommunityEdition/910beta1/programug/acldevg/runtime_doc_dev_0045.html">昇腾社区 CANN Runtime 文档</a></td>
</tr>
<tr>
<td align="left">CANN Runtime：单流捕获</td>
<td align="left">进一步查看底层捕获流程、Stream 同步限制、执行对象启动和资源释放</td>
<td align="left"><a href="https://www.hiascend.com/document/detail/zh/CANNCommunityEdition/910beta1/programug/acldevg/runtime_doc_dev_0030.html">昇腾社区 CANN Runtime 文档</a></td>
</tr>
<tr>
<td align="left">PyTorch 编译模式（torch.compile）</td>
<td align="left">对比 <code>inductor</code>、<code>npugraphs</code>、<code>npugraph_ex</code> 三种后端，并查看参数说明</td>
<td align="left"><a href="https://www.hiascend.com/document/detail/zh/Pytorch/2600/ptmoddevg/Frameworkfeatures/docs/zh/framework_feature_guide_pytorch/pytorch_compilation_mode.md">昇腾社区 PyTorch 文档</a></td>
</tr>
<tr>
<td align="left">NPUGraph</td>
<td align="left">查阅 <code>capture_begin()</code> / <code>capture_end()</code>、<code>make_graphed_callables</code>、<code>torch_npu.npu.graph</code> 等 API 细节</td>
<td align="left"><a href="https://www.hiascend.com/document/detail/zh/Pytorch/2600/ptmoddevg/Frameworkfeatures/docs/zh/framework_feature_guide_pytorch/pytorch_npugraph_desc.md">昇腾社区 PyTorch 文档</a></td>
</tr>
<tr>
<td align="left">npugraph_ex 快速上手</td>
<td align="left">查看 <code>backend="npugraph_ex"</code> 的最小样例和参数约束</td>
<td align="left"><a href="https://www.hiascend.com/document/detail/zh/Pytorch/2600/modthirdparty/torchairuseguide/docs/zh/npugraph_ex/quick_start.md">昇腾社区 TorchAir 文档</a></td>
</tr>
</tbody>
</table>
<br clear="all">


---

## 5. 本节小结

本节重点是理解 Aclgraph 为什么能降低重复执行场景中的 Host 侧开销，以及如何在 PyTorch 代码中启用这类能力。

- Aclgraph 通过捕获、实例化和重放稳定执行流，减少 Host 侧重复组织和提交任务的开销；
- 这类收益更适合固定 shape、短算子较多、重复执行的低时延场景；
- API 方式适合显式控制捕获边界，`torch.compile` 方式适合从模型入口启用后端能力；
- Pybind11 和 Torch Library 提供不同的自定义算子调用入口，提交到捕获 Stream 的 Kernel 任务均可随执行流一起被记录和重放；
- 首次捕获与实例化存在准备成本，观察性能时应区分首次调用和稳定重放阶段。


---

## 6. 课后练习

1. （判断题）某个算子本身计算时间很长，Host 侧下发开销占比很低。即使将它所在的稳定计算区域交给 Aclgraph 捕获与重放，单次算子的数学计算量也不会因此减少。

2. （判断题）如果一段计算只运行一次，首次捕获、编译或实例化成本无法被后续重放摊薄，因此不能只根据“使用了捕获与重放路径”判断一定更快。

3. （判断题）Aclgraph 捕获记录的是一段稳定执行流。若后续输入 shape、执行路径或关键张量地址频繁变化，捕获结果复用价值会下降。

4. （单选题）下面哪个场景最能体现 Aclgraph 的典型收益？  
   A. 每次只运行一遍的大型离线脚本  
   B. 固定 shape 的推理循环中反复执行一组短算子  
   C. 每轮都会改变模型结构的调试代码  
   D. 主要耗时集中在单个长时间运行算子的任务

5. （单选题）在分析一段代码是否适合 Aclgraph 捕获与重放时，最先应该判断的是：  
   A. 是否能把所有 Python 代码都删除  
   B. 是否存在稳定且会重复执行的 NPU 计算区域  
   C. 是否一定使用 `npugraph_ex` 后端  
   D. 是否可以不再检查输出正确性

6. （单选题）希望理解 Aclgraph 的捕获生命周期时，下面哪个入口最能直接体现“开始捕获、结束捕获、重放”的过程？  
   A. `capture_begin()` / `capture_end()` 与 `replay()`  
   B. `torch.randn()`  
   C. `backend="inductor"`  
   D. `optimizer.step()`

7. （单选题）某个模型整体包含动态分支，但其中一个子模块的输入 shape 和执行路径都比较稳定。更合适的处理思路是：  
   A. 放弃所有捕获与重放能力  
   B. 优先将稳定子模块封装为可复用的可调用对象，外围动态逻辑保持 Eager  
   C. 将动态分支强行放入同一个静态捕获区域中  
   D. 每轮为该子模块创建不同 shape 的输入

8. （多选题）关于框架入口、自定义算子和 Aclgraph 重放，下列说法正确的是：  
   A. 可以通过 `torch.compile(..., backend="npugraphs")` 或 `backend="npugraph_ex"` 从模型入口启用相关后端  
   B. 首次调用可能包含捕获、编译、实例化或缓存准备  
   C. 自定义算子将 Kernel 提交到正在捕获的 NPU Stream 后，其 Device 任务可以随执行流一起被记录  
   D. Pybind11 和 Torch Library 会改变自定义 Kernel 的数学计算逻辑

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/06.02_answer.txt
